In [0]:
df_silver = spark.sql("SELECT * FROM parquet.`abfss://silver@carsalesproject23.dfs.core.windows.net/carsales`")
df_silver.display()

In [0]:
df_branch = spark.sql("SELECT * FROM cars_catalog.gold.dim_branch")
df_model = spark.sql("SELECT * FROM cars_catalog.gold.dim_model")
df_date = spark.sql("SELECT * FROM cars_catalog.gold.dim_date")
df_dealer = spark.sql("SELECT * FROM cars_catalog.gold.dim_dealer")

### Bringing surrogate keys to fact table

In [0]:
df_fact = df_silver.join(df_branch, df_silver['Branch_ID'] == df_branch['branch_id'], 'left') \
                   .join(df_model, df_silver['Model_ID'] == df_model['model_id'], 'left') \
                   .join(df_date, df_silver['Date_ID'] == df_date['date_id'], 'left') \
                   .join(df_dealer, df_silver['Dealer_ID'] == df_dealer['dealer_id'], 'left') \
                   .select(
                       df_silver['Revenue'],
                       df_silver['Units_Sold'],
                       df_silver['rev_per_unit'],
                       df_branch['dim_branch_key'],
                       df_model['dim_model_key'],
                       df_date['dim_date_key'],
                       df_dealer['dim_dealer_key']
                   )
display(df_fact)

### Writing Fact Table

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists('factsales'):
    deltatbl = DeltaTable.forName('spark', 'cars_catalog.gold.factsales')
    deltatbl.alias('tgt').merge(
        df_fact.alias('src'),
        (
            'tgt.dim_date_key = src.dim_date_key AND '
            'tgt.dim_branch_key = src.dim_branch_key AND '
            'tgt.dim_dealer_key = src.dim_dealer_key AND '
            'tgt.dim_model_key = src.dim_model_key'
        )
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    df_fact.write.format('delta') \
        .mode('overwrite') \
        .option('path', 'abfss://gold@carsalesproject23.dfs.core.windows.net/factsales') \
        .saveAsTable('cars_catalog.gold.factsales')

In [0]:
%sql
select * from cars_catalog.gold.factsales